# MediGuard — fine-tuning TrOCR for handwritten prescriptionsFine-tunes `microsoft/trocr-small-handwritten` on doctors' handwriting so therecogniser feeding MediGuard's lexicon snapping is adapted to the domain.**Runtime → Change runtime type → T4 GPU** before running anything.## What this producesTwo artefacts, both consumed by code that already exists in the repo:| Artefact | Goes to | Used by ||---|---|---|| `trocr-finetuned/` | `apps/ml-service/data/trocr-finetuned/` | `MEDIGUARD_TROCR_MODEL` at inference || `handwriting/` (crops + `labels.jsonl`) | `apps/ml-service/data/testset/handwriting/` | `python scripts/ablation.py --trocr` |The held-out split is exported deliberately: `app/ocr/handwriting.py::ablate_checkpoints`reads it to produce the **before vs after CER table**, which is the number themigration plan asks for. Training without exporting it leaves you unable toprove the fine-tuning helped.## Why small, not base`trocr-base-handwritten` is ~3x the parameters. Inference runs on a free-tier**CPU** container, and a prescription can produce dozens of word crops — basewould blow the request budget. The accuracy gap is largely recovered by thelexicon snapping downstream, which is the actual design bet of this project.

## 1. Check the GPU

In [ ]:
import subprocessprint(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or      "No GPU. Runtime -> Change runtime type -> T4 GPU, then restart.")

## 2. Install dependenciesPinned to the versions the ML service runs, so a checkpoint trained here loads there.

In [ ]:
!pip install -q "transformers==4.49.0" "datasets>=2.19" "evaluate" "jiwer" "rapidfuzz==3.11.0" "sentencepiece==0.2.2" "accelerate>=0.30"print("done")

## 3. Kaggle credentialsKaggle → your avatar → **Settings** → **API** → *Create New Token*. Thatdownloads `kaggle.json`. Run the cell and upload it.

In [ ]:
import os, jsonfrom google.colab import filesif not os.path.exists("/root/.kaggle/kaggle.json"):    print("Upload kaggle.json:")    up = files.upload()    os.makedirs("/root/.kaggle", exist_ok=True)    with open("/root/.kaggle/kaggle.json", "wb") as fh:        fh.write(next(iter(up.values())))    os.chmod("/root/.kaggle/kaggle.json", 0o600)!pip install -q kaggleprint("kaggle ready:", json.load(open("/root/.kaggle/kaggle.json"))["username"])

## 4. Download the datasets**Verify these slugs before running** — Kaggle datasets get renamed and reuploaded.Search Kaggle for "doctor handwritten prescription" and paste the correct`owner/dataset-name` below. The search cell lists candidates for you.IAM is included as a general-handwriting mirror. It broadens the model beyondthe prescription set, which matters because the prescription datasets are smalland a model trained only on them overfits to a handful of writers.

In [ ]:
!kaggle datasets list -s "doctor handwritten prescription" | head -20print()!kaggle datasets list -s "iam handwriting" | head -10

In [ ]:
# Paste the slugs you confirmed above.PRESCRIPTION_SLUG = "mamun1113/doctors-handwritten-prescription-bd-dataset"IAM_SLUG          = "naderabdalghani/iam-handwritten-forms-dataset"  # optionalUSE_IAM           = False   # flip on once the prescription arm trains cleanlyimport pathlibDATA = pathlib.Path("/content/data"); DATA.mkdir(exist_ok=True)!kaggle datasets download -d {PRESCRIPTION_SLUG} -p /content/data --unzip -qif USE_IAM:    !kaggle datasets download -d {IAM_SLUG} -p /content/data --unzip -qfor p in sorted(DATA.rglob("*"))[:25]:    print(p.relative_to(DATA))

## 5. Build (image, text) pairsPrescription datasets are usually laid out as either* `Training/<LABEL>/img001.png` — the label is the folder name, or* a CSV mapping filename to label.Both are handled. **Inspect the listing above and set `LAYOUT` accordingly** —this is the one cell that genuinely needs your eyes, because getting the labelmapping wrong trains the model on noise and every downstream number is garbage.

In [ ]:
import csv, pathlib, randomfrom PIL import ImageLAYOUT = "folder"       # "folder" or "csv"CSV_PATH = ""           # only for LAYOUT="csv"CSV_IMAGE_COL, CSV_LABEL_COL = "image", "label"IMAGE_ROOT = DATA       # narrow this if the archive nests things deeplypairs = []if LAYOUT == "folder":    for img in IMAGE_ROOT.rglob("*"):        if img.suffix.lower() not in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:            continue        label = img.parent.name.strip()        # Skip split directories being mistaken for labels.        if label.lower() in {"training", "train", "test", "val", "validation", "data", "images"}:            continue        pairs.append((img, label))else:    with open(CSV_PATH, newline="", encoding="utf-8") as fh:        for row in csv.DictReader(fh):            matches = list(IMAGE_ROOT.rglob(row[CSV_IMAGE_COL]))            if matches:                pairs.append((matches[0], row[CSV_LABEL_COL].strip()))random.seed(13)random.shuffle(pairs)print("pairs:", len(pairs))print("distinct labels:", len({t for _, t in pairs}))for img, txt in pairs[:10]:    print("  %-46s %s" % (img.name, txt))

Sanity-check a few crops visually. If these do not look like handwritten medicine names, the layout setting above is wrong.

In [ ]:
import matplotlib.pyplot as pltfig, axes = plt.subplots(1, 6, figsize=(18, 3))for ax, (img, txt) in zip(axes, pairs[:6]):    ax.imshow(Image.open(img).convert("RGB")); ax.set_title(txt, fontsize=9); ax.axis("off")plt.tight_layout(); plt.show()

## 6. SplitThe **test split is exported to the repo** and never trained on — it is what`scripts/ablation.py --trocr` scores both checkpoints against. Splitting *bylabel* keeps the same medicine name from appearing in both train and test,which would otherwise inflate the fine-tuned number and make the ablationmeaningless.

In [ ]:
labels = sorted({t for _, t in pairs})random.seed(13); random.shuffle(labels)n_test = max(1, int(0.15 * len(labels)))n_val  = max(1, int(0.10 * len(labels)))test_labels  = set(labels[:n_test])val_labels   = set(labels[n_test:n_test + n_val])train = [(i, t) for i, t in pairs if t not in test_labels and t not in val_labels]val   = [(i, t) for i, t in pairs if t in val_labels]test  = [(i, t) for i, t in pairs if t in test_labels]print("train %d   val %d   test %d   (disjoint labels)" % (len(train), len(val), len(test)))

## 7. Model and processor

In [ ]:
import torchfrom transformers import TrOCRProcessor, VisionEncoderDecoderModelBASE = "microsoft/trocr-small-handwritten"processor = TrOCRProcessor.from_pretrained(BASE)model = VisionEncoderDecoderModel.from_pretrained(BASE).to(    "cuda" if torch.cuda.is_available() else "cpu")# VisionEncoderDecoder does not infer these, and generation is silently wrong# without them: no start token means the decoder never begins properly.model.config.decoder_start_token_id = processor.tokenizer.cls_token_idmodel.config.pad_token_id           = processor.tokenizer.pad_token_idmodel.config.eos_token_id           = processor.tokenizer.sep_token_idmodel.config.vocab_size             = model.config.decoder.vocab_sizemodel.generation_config.max_length          = 32   # medicine names are shortmodel.generation_config.num_beams           = 4model.generation_config.early_stopping      = Truemodel.generation_config.no_repeat_ngram_size = 3model.generation_config.length_penalty      = 2.0print("params: %.1fM" % (model.num_parameters() / 1e6))

## 8. Dataset

In [ ]:
from torch.utils.data import DatasetMAX_TARGET = 32class CropDataset(Dataset):    def __init__(self, rows, processor):        self.rows, self.processor = rows, processor    def __len__(self):        return len(self.rows)    def __getitem__(self, i):        path, text = self.rows[i]        image = Image.open(path).convert("RGB")        pixel_values = self.processor(image, return_tensors="pt").pixel_values[0]        ids = self.processor.tokenizer(            text, padding="max_length", max_length=MAX_TARGET, truncation=True).input_ids        # -100 is ignored by the loss; without this the model is rewarded for        # predicting padding and converges to emitting blanks.        ids = [t if t != self.processor.tokenizer.pad_token_id else -100 for t in ids]        return {"pixel_values": pixel_values, "labels": torch.tensor(ids)}train_ds, val_ds = CropDataset(train, processor), CropDataset(val, processor)print(len(train_ds), len(val_ds))

## 9. CER metric, logged every epoch

In [ ]:
import numpy as npfrom rapidfuzz.distance import Levenshteindef char_error_rate(ref, hyp):    ref = (ref or "").strip()    if not ref:        return 0.0 if not (hyp or "").strip() else 1.0    return Levenshtein.distance(ref, hyp or "") / len(ref)def compute_metrics(pred):    ids = pred.predictions    labels = pred.label_ids.copy()    labels[labels == -100] = processor.tokenizer.pad_token_id    hyps = processor.batch_decode(ids, skip_special_tokens=True)    refs = processor.batch_decode(labels, skip_special_tokens=True)    scores = [char_error_rate(r, h) for r, h in zip(refs, hyps)]    exact = np.mean([r.strip().lower() == h.strip().lower() for r, h in zip(refs, hyps)])    return {"cer": float(np.mean(scores)), "exact_match": float(exact)}

## 10. Baseline firstMeasure the **off-the-shelf** model before training. Without this number thefine-tuning has nothing to be compared against, and the whole ablation isunfalsifiable.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collatorargs = Seq2SeqTrainingArguments(    output_dir="/content/trocr-out",    predict_with_generate=True,    eval_strategy="epoch",    save_strategy="epoch",    per_device_train_batch_size=8,    per_device_eval_batch_size=8,    num_train_epochs=8,    learning_rate=5e-5,    warmup_ratio=0.1,    fp16=torch.cuda.is_available(),    logging_steps=25,    save_total_limit=2,    load_best_model_at_end=True,    metric_for_best_model="cer",    greater_is_better=False,    report_to=[],)trainer = Seq2SeqTrainer(    model=model, args=args,    train_dataset=train_ds, eval_dataset=val_ds,    data_collator=default_data_collator,    compute_metrics=compute_metrics,)baseline = trainer.evaluate()print("BASELINE  CER %.4f   exact %.4f" % (baseline["eval_cer"], baseline["eval_exact_match"]))

## 11. TrainCER per epoch appears in the table below — that is the curve the plan asks you to log.

In [ ]:
trainer.train()final = trainer.evaluate()print()print("BASELINE   CER %.4f   exact %.4f" % (baseline["eval_cer"], baseline["eval_exact_match"]))print("FINE-TUNED CER %.4f   exact %.4f" % (final["eval_cer"], final["eval_exact_match"]))delta = baseline["eval_cer"] - final["eval_cer"]print("improvement: %.4f CER (%.1f%% relative)" % (delta, 100 * delta / max(baseline["eval_cer"], 1e-9)))

## 12. ExportWrites both artefacts into one folder and zips it. Unzip into`apps/ml-service/data/` so the paths line up with what the repo already expects.

In [ ]:
import json, shutil, pathlibOUT = pathlib.Path("/content/mediguard_trocr"); OUT.mkdir(exist_ok=True)ckpt = OUT / "trocr-finetuned"model.save_pretrained(ckpt)processor.save_pretrained(ckpt)# Held-out crops in the exact shape ablate_checkpoints() reads:#   data/testset/handwriting/labels.jsonl -> {"image": <filename>, "text": <label>}hw = OUT / "testset" / "handwriting"hw.mkdir(parents=True, exist_ok=True)with (hw / "labels.jsonl").open("w", encoding="utf-8") as fh:    for n, (path, text) in enumerate(test):        name = "crop_%04d%s" % (n, pathlib.Path(path).suffix.lower())        shutil.copy(path, hw / name)        fh.write(json.dumps({"image": name, "text": text}) + "\n")(OUT / "training_report.json").write_text(json.dumps({    "base_model": BASE,    "baseline_cer": baseline["eval_cer"],    "finetuned_cer": final["eval_cer"],    "baseline_exact_match": baseline["eval_exact_match"],    "finetuned_exact_match": final["eval_exact_match"],    "train_size": len(train), "val_size": len(val), "test_size": len(test),    "epochs": args.num_train_epochs,    "log_history": trainer.state.log_history,}, indent=2), encoding="utf-8")shutil.make_archive("/content/mediguard_trocr", "zip", OUT)print("zipped:", pathlib.Path("/content/mediguard_trocr.zip").stat().st_size / 1e6, "MB")

In [ ]:
from google.colab import filesfiles.download("/content/mediguard_trocr.zip")

## 13. Install locallyUnzip into `apps/ml-service/data/` so you end up with:```apps/ml-service/data/  trocr-finetuned/               <- checkpoint  testset/handwriting/    labels.jsonl    crop_0000.png ...  training_report.json```Then produce the before/after table:```bashcd apps/ml-servicepython scripts/ablation.py --trocr```That prints CER **and** name accuracy after lexicon snapping for bothcheckpoints — the second column is the one that matters, because it measureswhat the user actually sees. A modest CER gain can produce a large accuracy gainonce snapping constrains the output to real medicine names, and that gap is theproject's central claim.To serve the fine-tuned checkpoint:```bashMEDIGUARD_TROCR_MODEL=data/trocr-finetuned \  python -m uvicorn app.main:app --port 8000```**Commit the report and the test split, not the checkpoint.** `training_report.json`and `testset/handwriting/` are small and make the numbers reproducible; thecheckpoint is hundreds of MB and belongs in the Space image or a releaseartefact instead.